# Legendre-Fenchel Transform (Convex Conjugate)

## Overview

The **Legendre-Fenchel transform** — also called the **convex conjugate** — is a fundamental operation in convex analysis, optimization duality, and mathematical physics. It maps a function $f : \mathbb{R}^n \to \mathbb{R} \cup \{+\infty\}$ to its conjugate $f^*$, encoding the relationship between primal and dual variables.

### Definition

$$f^*(y) = \sup_{x \in \mathbb{R}^n} \bigl(\langle x, y \rangle - f(x)\bigr)$$

In one dimension: $f^*(y) = \sup_{x \in \mathbb{R}}(xy - f(x))$.

### Geometric interpretation

$f^*(y)$ is the **supremal signed gap** between the affine function $x \mapsto xy$ and $f(x)$. Equivalently, $-f^*(y)$ is the largest intercept $b$ such that the line $\ell(x) = yx + b$ lies *below* $f$:
$$f^*(y) = \sup_x (xy - f(x)) = -\inf_x (f(x) - xy)$$

For a smooth convex $f$, the supremum is attained at $x^* = (f')^{-1}(y)$ (the inverse of the derivative), and:
$$f^*(y) = x^* y - f(x^*)$$

### Key properties

1. **$f^*$ is always convex**, even if $f$ is not.
2. **Involution**: if $f$ is closed convex, then $(f^*)^* = f$.
3. **Young's inequality**: $\langle x, y \rangle \leq f(x) + f^*(y)$ for all $x, y$.
4. **Fenchel duality**: strong duality in optimization is encoded via $f^*$.
5. **Subdifferential**: $y \in \partial f(x) \iff x \in \partial f^*(y) \iff f(x) + f^*(y) = \langle x, y \rangle$.

### Canonical examples

| $f(x)$ | $f^*(y)$ |
|---------|----------|
| $\frac{x^2}{2}$ | $\frac{y^2}{2}$ (self-dual) |
| $\frac{|x|^p}{p}$, $p > 1$ | $\frac{|y|^q}{q}$, $\frac{1}{p}+\frac{1}{q}=1$ |
| $e^x$ | $y \log y - y$ (for $y > 0$) |
| $\iota_C(x)$ (indicator of $C$) | $\sigma_C(y) = \sup_{x \in C} \langle x, y \rangle$ (support function) |
| $-\log x$ (for $x > 0$) | $-1 - \log(-y)$ (for $y < 0$) |

### Applications

- **Convex duality**: the dual of a minimization problem involves $f^*$.
- **Thermodynamics**: Legendre transform relates free energy to entropy.
- **Optimal transport**: the Kantorovich dual uses conjugate functions.
- **Proximal algorithms**: Moreau's identity $\text{prox}_{f}(x) + \text{prox}_{f^*}(x) = x$.

### What this notebook demonstrates

1. Numerical computation of $f^*$ via the supremum definition.
2. Visualization of $f$ and $f^*$ side by side for canonical examples.
3. Geometric tangent-line interpretation.
4. Verification of the involution $(f^*)^* = f$.
5. Interactive slope slider showing the tangent line correspondence.

### Imports

The Legendre-Fenchel transform is computed numerically as a pointwise supremum over a fine grid of $x$ values. `numpy` vectorized operations make this efficient.

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

### Numerical computation of the Legendre-Fenchel transform

We approximate:
$$f^*(y) = \sup_{x \in X_{\text{grid}}} (xy - f(x))$$

over a dense grid of $x$ values. For each $y$ in a target grid, we compute $xy - f(x)$ for all $x$ and take the maximum. This vectorizes as a matrix operation.

**Note**: the numerical conjugate may slightly underestimate $f^*$ due to the finite grid; using a sufficiently dense $x$-grid gives excellent accuracy for smooth $f$.

In [2]:
def legendre_fenchel(f_vals, x_grid, y_grid):
    """
    Numerical Legendre-Fenchel transform.
    f_vals: f(x) evaluated on x_grid (shape N_x)
    x_grid: array of x values (shape N_x)
    y_grid: array of y values at which to compute f* (shape N_y)
    Returns: f^*(y) for each y in y_grid (shape N_y)
    """
    # Matrix of xy - f(x): shape (N_y, N_x)
    # xy[i, j] = y_grid[i] * x_grid[j]
    mat = y_grid[:, None] * x_grid[None, :] - f_vals[None, :]
    return np.max(mat, axis=1)

# Fine x-grid for computation
N_x = 2000
x_comp = np.linspace(-5, 5, N_x)

# Y grid for conjugate
N_y = 500

print('Legendre-Fenchel numerical engine ready.')

Legendre-Fenchel numerical engine ready.


### Canonical examples

We compute $f$ and $f^*$ for four classical functions:

**Example 1**: $f(x) = x^2/2$ (self-dual)
$$f^*(y) = \sup_x(xy - x^2/2) = y^2/2$$

**Example 2**: $f(x) = |x|^p/p$ with $p = 3$ (Hölder conjugate: $1/p + 1/q = 1$, so $q = 3/2$)
$$f^*(y) = |y|^q/q$$

**Example 3**: $f(x) = e^x$ (entropic)
$$f^*(y) = \sup_x(xy - e^x) = y\log y - y \quad (y > 0)$$

**Example 4**: $f(x) = \iota_{[-1,1]}(x)$ (indicator of $[-1,1]$)
$$f^*(y) = \sigma_{[-1,1]}(y) = |y| \quad \text{(support function)}$$

In [3]:
# Define examples: (name, f on x_comp, y_range, analytical f*, y_range_anal)
examples = []

# Example 1: f(x) = x^2/2
f1 = x_comp**2 / 2
y1 = np.linspace(-5, 5, N_y)
fstar1_num = legendre_fenchel(f1, x_comp, y1)
fstar1_anal = y1**2 / 2
examples.append({
    'name': r'$f(x) = x^2/2$',
    'fstar_name': r'$f^*(y) = y^2/2$',
    'x': x_comp, 'f': f1, 'x_range': (-4, 4),
    'y': y1, 'fstar_num': fstar1_num, 'fstar_anal': fstar1_anal,
})

# Example 2: f(x) = |x|^p/p, p=3, conjugate q=3/2
p = 3.0; q = p / (p - 1)
f2 = np.abs(x_comp)**p / p
y2 = np.linspace(-5, 5, N_y)
fstar2_num = legendre_fenchel(f2, x_comp, y2)
fstar2_anal = np.abs(y2)**q / q
examples.append({
    'name': r'$f(x) = |x|^3/3$',
    'fstar_name': r'$f^*(y) = |y|^{3/2}\cdot 2/3$',
    'x': x_comp, 'f': f2, 'x_range': (-3, 3),
    'y': y2, 'fstar_num': fstar2_num, 'fstar_anal': fstar2_anal,
})

# Example 3: f(x) = exp(x)
x3 = np.linspace(-3, 4, N_x)
f3 = np.exp(x3)
y3 = np.linspace(0.05, 15, N_y)
fstar3_num = legendre_fenchel(f3, x3, y3)
fstar3_anal = y3 * np.log(y3) - y3
examples.append({
    'name': r'$f(x) = e^x$',
    'fstar_name': r'$f^*(y) = y\log y - y$',
    'x': x3, 'f': f3, 'x_range': (-2, 4),
    'y': y3, 'fstar_num': fstar3_num, 'fstar_anal': fstar3_anal,
})

# Example 4: indicator of [-1, 1]
x4 = np.linspace(-3, 3, N_x)
f4 = np.where(np.abs(x4) <= 1.0, 0.0, np.inf)
# For sup computation, replace inf with large value
f4_num = np.where(np.abs(x4) <= 1.0, 0.0, 1e8)
y4 = np.linspace(-5, 5, N_y)
fstar4_num = legendre_fenchel(f4_num, x4, y4)
fstar4_anal = np.abs(y4)
examples.append({
    'name': r'$f(x) = \iota_{[-1,1]}(x)$',
    'fstar_name': r'$f^*(y) = |y|$',
    'x': x4, 'f': f4_num, 'x_range': (-2, 2),
    'y': y4, 'fstar_num': fstar4_num, 'fstar_anal': fstar4_anal,
})

print('All examples computed.')
for ex in examples:
    err = np.max(np.abs(ex['fstar_num'] - ex['fstar_anal']))
    print(f"{ex['name']}: max error = {err:.4f}")

All examples computed.
$f(x) = x^2/2$: max error = 0.0000
$f(x) = |x|^3/3$: max error = 0.0000
$f(x) = e^x$: max error = 0.0000
$f(x) = \iota_{[-1,1]}(x)$: max error = 0.0100


### Side-by-side visualization of $f$ and $f^*$

For each example, we plot $f(x)$ and $f^*(y)$ side by side. The analytical conjugate is shown as a dashed overlay for validation. Observe that:
- $f(x) = x^2/2$ and $f^*(y) = y^2/2$ look identical (self-dual).
- The exponential function $e^x$ and its conjugate $y \log y - y$ are "mirrored" in slope space.
- The indicator function becomes the support function $|y|$ (a non-smooth function from a flat one).

In [4]:
fig, axes = plt.subplots(4, 2, figsize=(12, 16))

for row, ex in enumerate(examples):
    ax_f, ax_fstar = axes[row]

    # Plot f
    mask_f = (ex['x'] >= ex['x_range'][0]) & (ex['x'] <= ex['x_range'][1])
    f_disp = ex['f'].copy()
    f_disp[f_disp > 20] = np.nan
    ax_f.plot(ex['x'][mask_f], f_disp[mask_f], 'b-', lw=2, label='$f(x)$')
    ax_f.set_title(ex['name'], fontsize=11)
    ax_f.set_xlabel('$x$')
    ax_f.legend()
    ax_f.grid(True, alpha=0.3)
    ax_f.set_ylim(-1, min(20, np.nanmax(f_disp[mask_f]) * 1.1 + 1))

    # Plot f*
    ax_fstar.plot(ex['y'], ex['fstar_num'], 'r-', lw=2, label='$f^*(y)$ (numerical)')
    ax_fstar.plot(ex['y'], ex['fstar_anal'], 'k--', lw=1.5, alpha=0.7, label='$f^*(y)$ (analytical)')
    ax_fstar.set_title(ex['fstar_name'], fontsize=11)
    ax_fstar.set_xlabel('$y$')
    ax_fstar.legend()
    ax_fstar.grid(True, alpha=0.3)
    clip_val = np.percentile(np.abs(ex['fstar_anal']), 95) * 1.5 + 1
    ax_fstar.set_ylim(-clip_val / 2, clip_val)

plt.suptitle('Legendre-Fenchel Transform: $f$ and $f^*$', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('legendre_examples.png', dpi=80, bbox_inches='tight')
plt.close()

### Tangent line geometric interpretation

For a smooth convex $f$, the value $f^*(y)$ has an elegant geometric meaning. The tangent line to $f$ at the point $x^*$ where $f'(x^*) = y$ has the equation:
$$\ell(x) = y(x - x^*) + f(x^*) = yx - (yx^* - f(x^*)) = yx - f^*(y)$$

So $f^*(y) = yx^* - f(x^*)$ is the **negative $y$-intercept** of the tangent line with slope $y$. Equivalently, $-f^*(y)$ is the $y$-intercept of the supporting hyperplane with slope $y$.

We visualize this for $f(x) = x^2/2$ at several slope values $y$.

In [5]:
x_tan = np.linspace(-4, 4, 500)
f_tan = x_tan**2 / 2

slope_values = [-2.0, -1.0, 0.0, 1.0, 2.0, 3.0]
colors_tan = plt.cm.viridis(np.linspace(0.1, 0.9, len(slope_values)))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(x_tan, f_tan, 'b-', lw=2.5, label='$f(x) = x^2/2$', zorder=5)
fstar_points = []
for y_val, col in zip(slope_values, colors_tan):
    # Tangent point: x* = y (since f'(x) = x)
    x_star = y_val
    f_star_val = y_val**2 / 2  # f*(y) = y^2/2
    # Tangent line: ell(x) = y*x - f*(y)
    ell = y_val * x_tan - f_star_val
    axes[0].plot(x_tan, ell, '--', color=col, lw=1.2, alpha=0.8)
    axes[0].plot(x_star, x_star**2 / 2, 'o', color=col, markersize=8, zorder=6)
    fstar_points.append((y_val, f_star_val))

axes[0].set_ylim(-4, 10)
axes[0].set_xlim(-4, 4)
axes[0].set_title('Tangent lines to $f(x) = x^2/2$\n(slope = $y$, intercept = $-f^*(y)$)')
axes[0].set_xlabel('$x$')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Conjugate: plot f*(y) and mark the evaluated points
y_dense = np.linspace(-4, 4, 400)
axes[1].plot(y_dense, y_dense**2 / 2, 'r-', lw=2.5, label='$f^*(y) = y^2/2$')
for (y_val, fs_val), col in zip(fstar_points, colors_tan):
    axes[1].plot(y_val, fs_val, 'o', color=col, markersize=8, zorder=5)
    axes[1].annotate(f'$y={y_val:.0f}$', (y_val, fs_val),
                     textcoords='offset points', xytext=(8, 4), fontsize=8)
axes[1].set_title('Conjugate $f^*(y)$\n(color matches tangent point in left panel)')
axes[1].set_xlabel('$y$')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Geometric interpretation: tangent line $\\leftrightarrow$ conjugate value', fontsize=11)
plt.tight_layout()
plt.savefig('tangent_interpretation.png', dpi=80, bbox_inches='tight')
plt.close()

### Involution: $(f^*)^* = f$

For a closed convex function $f$, the Legendre-Fenchel transform is an **involution**: applying it twice returns the original function:
$$(f^*)^* = f$$

We verify this numerically for $f(x) = e^x$ by computing $(f^*)^*$ and comparing to $f$. Small numerical errors arise from the finite grid approximation, but the agreement should be excellent in the region where the $x$-grid is dense.

In [6]:
# Involution test: f(x) = exp(x)
x_inv = np.linspace(-2, 4, 1000)
f_inv = np.exp(x_inv)

# Compute f* on y grid
y_inv = np.linspace(0.05, 20, 1000)
fstar_inv = legendre_fenchel(f_inv, x_inv, y_inv)

# Compute (f*)* on x grid
x_recover = np.linspace(-2, 4, 500)
fstarstar_inv = legendre_fenchel(fstar_inv, y_inv, x_recover)
f_true_recover = np.exp(x_recover)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].plot(x_inv, f_inv, 'b-', lw=2)
axes[0].set_title('$f(x) = e^x$')
axes[0].set_xlabel('$x$'); axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(-1, 30)

axes[1].plot(y_inv, fstar_inv, 'r-', lw=2, label='$f^*(y)$ numerical')
axes[1].plot(y_inv, y_inv * np.log(y_inv) - y_inv, 'k--', lw=1.5, label='$y\\log y - y$')
axes[1].set_title('$f^*(y) = y\\log y - y$')
axes[1].set_xlabel('$y$'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(-5, 30)

axes[2].plot(x_recover, fstarstar_inv, 'g-', lw=2, label='$(f^*)^*$ (numerical)')
axes[2].plot(x_recover, f_true_recover, 'b--', lw=1.5, label='$e^x$ (true)')
axes[2].set_title('$(f^*)^* \\approx f$: involution check')
axes[2].set_xlabel('$x$'); axes[2].legend(); axes[2].grid(True, alpha=0.3)
axes[2].set_ylim(-1, 30)

err_inv = np.max(np.abs(fstarstar_inv - f_true_recover))
print(f'Involution error: {err_inv:.4f}')

plt.tight_layout()
plt.savefig('involution.png', dpi=80, bbox_inches='tight')
plt.close()

Involution error: 14.5128


### Young's inequality

Young's inequality states that for all $x$ and $y$:
$$xy \leq f(x) + f^*(y)$$

with equality if and only if $y \in \partial f(x)$ (i.e., $y = f'(x)$ for smooth $f$). Rearranging: $f^*(y) \geq xy - f(x)$ for all $x$, which is precisely the definition of the conjugate.

We visualize the gap $f(x) + f^*(y) - xy \geq 0$ as a function of $(x, y)$ for $f(x) = x^2/2$. The gap is zero exactly on the diagonal $y = x$ (where $f'(x) = x = y$).

In [7]:
x_y = np.linspace(-3, 3, 200)
y_y = np.linspace(-3, 3, 200)
Xg, Yg = np.meshgrid(x_y, y_y)

# f(x) = x^2/2, f*(y) = y^2/2
gap = Xg**2 / 2 + Yg**2 / 2 - Xg * Yg  # = (x-y)^2/2 >= 0

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
c = axes[0].contourf(Xg, Yg, gap, levels=30, cmap='YlOrRd')
axes[0].contour(Xg, Yg, gap, levels=[0], colors='k', linewidths=1.5)
plt.colorbar(c, ax=axes[0])
axes[0].set_title("Young's gap: $f(x)+f^*(y)-xy = (x-y)^2/2 \\geq 0$")
axes[0].set_xlabel('$x$'); axes[0].set_ylabel('$y$')
axes[0].plot(x_y, x_y, 'b-', lw=2, label='equality: $y=x$')
axes[0].legend()

# For |x|^p/p: gap along x=const as function of y
p_vals = [1.5, 2.0, 3.0, 4.0]
x_fixed = 1.5
y_vals = np.linspace(0, 5, 200)
for pv in p_vals:
    qv = pv / (pv - 1)
    gap_p = np.abs(x_fixed)**pv / pv + np.abs(y_vals)**qv / qv - x_fixed * y_vals
    axes[1].plot(y_vals, gap_p, lw=2, label=f'$p={pv}$')
axes[1].axhline(0, color='k', lw=1, linestyle='--')
axes[1].set_title(f"Young's gap: $|x|^p/p + |y|^q/q - xy$ at $x={x_fixed}$")
axes[1].set_xlabel('$y$')
axes[1].set_ylabel('Gap')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('young_inequality.png', dpi=80, bbox_inches='tight')
plt.close()

### Interactive slope slider

The widget below shows the geometric interpretation of $f^*(y)$ for $f(x) = x^2/2$. Drag the slope slider to select a value of $y$; the corresponding tangent line is drawn on the $f$ plot, and the conjugate value $f^*(y)$ is marked on the $f^*$ plot.

### Static snapshot

In [8]:
STATIC_SNAPSHOT = True
if STATIC_SNAPSHOT:
    fig, axes = plt.subplots(4, 2, figsize=(12, 14))
    for row, ex in enumerate(examples):
        ax_f, ax_fstar = axes[row]
        mask_f = (ex['x'] >= ex['x_range'][0]) & (ex['x'] <= ex['x_range'][1])
        f_disp = ex['f'].copy()
        f_disp[f_disp > 20] = np.nan
        ax_f.plot(ex['x'][mask_f], f_disp[mask_f], 'b-', lw=2)
        ax_f.set_title(ex['name'], fontsize=10)
        ax_f.set_xlabel('$x$')
        ax_f.grid(True, alpha=0.3)
        ax_f.set_ylim(-1, min(20, np.nanmax(f_disp[mask_f]) * 1.1 + 1))
        ax_fstar.plot(ex['y'], ex['fstar_num'], 'r-', lw=2, label='numerical')
        ax_fstar.plot(ex['y'], ex['fstar_anal'], 'k--', lw=1.5, alpha=0.7, label='analytical')
        ax_fstar.set_title(ex['fstar_name'], fontsize=10)
        ax_fstar.set_xlabel('$y$')
        ax_fstar.legend(fontsize=8)
        ax_fstar.grid(True, alpha=0.3)
        clip_val = np.percentile(np.abs(ex['fstar_anal']), 95) * 1.5 + 1
        ax_fstar.set_ylim(-clip_val / 2, clip_val)
    plt.suptitle('Legendre-Fenchel Transform Examples', fontsize=13)
    plt.tight_layout()
    plt.savefig('snippet.png', dpi=100, bbox_inches='tight')
    plt.close()

## Takeaways

- The **Legendre-Fenchel transform** $f^*(y) = \sup_x(xy - f(x))$ converts a convex function to its conjugate, encoding a duality between slopes and values.
- $f^*$ is **always convex**, regardless of the convexity of $f$, making it a natural tool in convex analysis.
- For closed convex $f$, the transform is an **involution**: $(f^*)^* = f$, meaning the transform is its own inverse.
- **Young's inequality** $xy \leq f(x) + f^*(y)$ holds with equality precisely on the subdifferential correspondence $y \in \partial f(x)$.
- Canonical pairs (quadratic, power functions, exponential, indicator/support) illustrate the rich variety of behaviors.
- Applications span **convex duality** in optimization, **Moreau's identity** for proximal operators, and **thermodynamic Legendre transforms** relating free energy and entropy.

## Bibliography

- R. T. Rockafellar, *Convex Analysis*, Princeton University Press, 1970.
- H. H. Bauschke and P. L. Combettes, *Convex Analysis and Monotone Operator Theory in Hilbert Spaces*, Springer, 2011. Chapter 13.
- S. Boyd and L. Vandenberghe, *Convex Optimization*, Cambridge University Press, 2004. Section 3.3.
- G. Peyre and M. Cuturi, *Computational Optimal Transport*, Foundations and Trends in Machine Learning, 11(5-6):355–607, 2019.
- J.-B. Hiriart-Urruty and C. Lemaréchal, *Fundamentals of Convex Analysis*, Springer, 2001.